In [ ]:
import pandas as pd
import numpy as np
import optuna

import matplotlib.pyplot as plt
import seaborn as sns

import os
import zipfile
import urllib.request
import warnings

from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder

import pprint

warnings.filterwarnings('ignore')

In [ ]:
data_dir = "../data"

In [ ]:
top_recs = 10

## Data downloading

In [ ]:
movies_data_dir_path = os.path.join(data_dir, "ml-1m")
zip_path = os.path.join(data_dir, "ml-1m.zip")
data_url = "http://files.grouplens.org/datasets/movielens/ml-1m.zip"

if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    
if not os.path.exists(movies_data_dir_path):
    print("there is no data. downloading")
    
    urllib.request.urlretrieve(data_url, zip_path)
    
    with zipfile.ZipFile(zip_path, "r") as zip_data:
        zip_data.extractall(data_dir)
        
        os.remove(zip_path)
        print(f'data is downloaded and extracted to {movies_data_dir_path}')
else:
    print(f'data already exists in {movies_data_dir_path}')

In [ ]:
ratings_path  = os.path.join(movies_data_dir_path, 'ratings.dat')
col_names = ['user_id', 'item_id', 'rating', 'timestamp']

df = pd.read_csv(ratings_path, sep='::', engine='python', names=col_names)
df.head()

## EDA

In [ ]:
print(f'interactions quantity: {df.shape}')
print(f'unique users: {len(df['user_id'].unique())}')
print(f'unique items: {len(df['item_id'].unique())}')

In [ ]:
def rating_quantity(main_data, group_by_col, func_col, func):
    group_df = main_data.groupby(group_by_col).agg(
        quantity=(func_col, func)
    ).reset_index()
    
    group_df = group_df.rename(columns={
        'quantity':f'{func_col}_quantity'
    })
    
    group_df = group_df.sort_values(by=f'{func_col}_quantity', ascending=False)
    
    return group_df

user_rating = rating_quantity(df, 'user_id', 'rating', 'size')
item_rating = rating_quantity(df, 'item_id', 'rating', 'size')

In [ ]:
item_rating.head(10)

In [ ]:
user_rating.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(
    data=user_rating,
    x='rating_quantity'
)

plt.title('Распределение количества оценок у пользователей')
plt.xlabel('Количество поставленных оценок')
plt.ylabel('Количество пользователей')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(
    data=item_rating,
    x='rating_quantity'
)

plt.title('Распределение количества оценок у фильмов')
plt.xlabel('Количество поставленных оценок')
plt.ylabel('Количество пользователей')
plt.show()

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
min_date = df['timestamp'].min()

df['last_watch_dt'] = df['timestamp'] - min_date
df['last_watch_dt'] = df['last_watch_dt'].apply(lambda x: int(str(x).split()[0]))
df = df.drop(columns='timestamp')

In [ ]:
df['last_watch_dt'].hist(bins=50, grid=False)
plt.show()

## Filtering data

In [ ]:
active_users = user_rating.loc[user_rating['rating_quantity'] >= 10, 'user_id'].unique()
active_films = item_rating.loc[item_rating['rating_quantity'] >= 10, 'item_id'].unique()

mask1 = df['rating'] >= 4
filtered_df = df[mask1].copy()

mask2 = (filtered_df['user_id'].isin(active_users)) & (filtered_df['item_id'].isin(active_films))
filtered_df = filtered_df[mask2].copy()

In [ ]:
print(f'old shape: {df.shape}')
print(f'filtered df shape: {filtered_df.shape}')

## Train test split

In [ ]:
test_frac = 0.2
max_test_items = 5
    
def get_test_indices(group):
    n_test = max(1, min(max_test_items, int(len(group) * test_frac)))
    return group.index[-n_test:]

filtered_df = filtered_df.sort_values(by=['user_id', 'last_watch_dt'])

test_indices = filtered_df.groupby('user_id').apply(get_test_indices).explode()
    
test_df = filtered_df.loc[test_indices].copy()
train_df = filtered_df.drop(test_indices).copy()

In [ ]:
train_users_ids = train_df['user_id'].unique()
train_item_ids = train_df['item_id'].unique()

test_df = test_df[test_df['user_id'].isin(train_users_ids)].copy()
test_df = test_df[test_df['item_id'].isin(train_item_ids)].copy()

In [ ]:
assert set(test_df['user_id'].unique()).issubset(set(train_df['user_id'].unique())), 'wrong user ids in test df'
assert set(test_df['item_id'].unique()).issubset(set(train_df['item_id'].unique())), 'wrong item ids in test df'

## Modeling

In [ ]:
from implicit.nearest_neighbours import bm25_weight

class ImplicitModel:
    def __init__(self, model):
        self.model = model
        self.trained = False
        
    def fit(self, train_df):
        self.item_encoder = LabelEncoder()
        self.user_encoder = LabelEncoder()
        self.item_encoder.fit(train_df['item_id'])
        self.user_encoder.fit(train_df['user_id'])
        
        self.train_ratings = self.create_matrix(train_df, ['item_id', 'user_id'])
        
        self.model.fit(self.train_ratings)
        self.trained = True
    
    def predict(self, test_df, top_k = 10):
        if not self.trained:
            raise ValueError("Model is not fitted")
        
        users_to_predict = test_df['user_id'].unique()
        encoded_users_to_predict = self.user_encoder.transform(users_to_predict)
        
        recs = self.model.recommend(
            encoded_users_to_predict, 
            self.train_ratings[encoded_users_to_predict], 
            N=top_k, 
            filter_already_liked_items=True
            )[0]
        
        user_recs = [self.item_encoder.inverse_transform(i) for i in recs]
        
        return user_recs
    
    def create_matrix(self, df, ids_list):
        encoded_item_ids = self.item_encoder.transform(df[ids_list[0]])
        encoded_user_ids = self.user_encoder.transform(df[ids_list[1]])
        
        n_items = len(self.item_encoder.classes_)
        n_users = len(self.user_encoder.classes_)
        
        matrix_shape = (n_users, n_items)
        weights = np.ones(len(encoded_user_ids), dtype=np.float32)
        
        sparse_matrix = csr_matrix(
            (weights, (encoded_user_ids, encoded_item_ids)), 
            shape=matrix_shape, 
            dtype=np.float32
        )
        
        return sparse_matrix

In [ ]:
base_params = {'factors': 64, 
               'regularization': 0.01, 
               'iterations': 15, 
               'random_state': 42
               }

In [ ]:
als_model = AlternatingLeastSquares(**base_params)
recsys_model = ImplicitModel(als_model)

recsys_model.fit(train_df)

In [ ]:
preds = recsys_model.predict(test_df, top_k=top_recs)

In [ ]:
pprint.pp(preds[:5])

## Evaluate model

In [ ]:
def recall_metric(ground_truth, preds):
    num_gt = len(ground_truth)
    if num_gt == 0:
        return 0
    
    relevant = list(set(ground_truth) & set(preds))
    num_relevant = len(relevant)
    
    return num_relevant / num_gt

def dcg_calc(scores):
    num = np.power(2, scores) - 1
    denom = np.log2(np.arange(scores.shape[0], dtype=np.float64) + 2)
    dcg = np.sum(num / denom)
    
    return dcg

def ndcg_metric(ground_truth, preds):
    num_preds = len(preds)
    if num_preds == 0:
        return 0
    
    relevant = np.array([1 if x in ground_truth else 0 for x in preds])
    
    dcg = dcg_calc(relevant)
    if dcg == 0.0:
        return 0.0
    
    ideal_relevant = np.zeros(num_preds)
    num_relevant = min(num_preds, len(ground_truth))
    ideal_relevant[:num_relevant] = 1
    
    ideal_dcg = dcg_calc(ideal_relevant)
    
    if ideal_dcg == 0.0:
        return 0.0
    
    normalized_dcg = dcg / ideal_dcg
    
    return normalized_dcg

def evaluate_model(test_df, preds_column, gt_column, top_k=top_recs):
    metrics = []
    
    for idx, row in test_df.iterrows():
        gt_items = row[gt_column]
        preds = row[preds_column][:top_k]
        
        ndcg = ndcg_metric(gt_items, preds)
        recall = recall_metric(gt_items, preds)
        
        metrics.append((ndcg, recall))
        
    mean_ndcg = np.mean([x[0] for x in metrics])
    mean_recall = np.mean([x[1] for x in metrics])
    
    return mean_ndcg, mean_recall

In [ ]:
test_df_grouped = test_df.groupby('user_id').apply(lambda x: list(x['item_id']))
test_df_grouped = test_df_grouped.reset_index(name='true_test_interactions')
test_df_grouped.head()

In [ ]:
preds_df = pd.DataFrame({
    'user_id': test_df['user_id'].unique(),
    'ials_preds': preds
})

test_df_grouped = test_df_grouped.merge(
    preds_df, 
    on='user_id', 
    how='left')

In [ ]:
ials_ndcg, ials_recall = evaluate_model(test_df_grouped, 'ials_preds', 'true_test_interactions', top_k=top_recs)

In [ ]:
metrics_df = pd.DataFrame(
    data={
        'metrics': [ials_ndcg, ials_recall]
    },
    index=[f'NDCG@{top_recs}', f'Recall@{top_recs}']
)

metrics_df

## Optimize params

In [ ]:
# optuna.logging.set_verbosity(optuna.logging.WARNING)

# def objective(trial):
#     factors = trial.suggest_categorical('factors', [32, 64, 128])
#     regularization = trial.suggest_float('regularization', 0.01, 0.5, log=True)
#     iterations = trial.suggest_int('iterations', 10, 30)
    
#     als_model = AlternatingLeastSquares(
#         factors=factors, 
#         regularization=regularization, 
#         iterations=iterations, 
#         random_state=42
#     )
    
#     recsys_model = ImplicitModel(als_model)
#     recsys_model.fit(train_df)
    
#     preds = recsys_model.predict(test_df, top_k=top_recs)
    
#     preds_df = pd.DataFrame({
#         'user_id': test_df['user_id'].unique(), 
#         'ials_preds': preds
#         })
    
#     test_df_grouped = test_df.groupby('user_id').apply(lambda x: list(x['item_id'])).reset_index(name='true_test_interactions')
#     test_df_grouped = test_df_grouped.merge(preds_df, on='user_id', how='left')
    
#     ndcg, _ = evaluate_model(test_df_grouped, 'ials_preds', 'true_test_interactions', top_k=top_recs)
#     return ndcg

# print("optimizing params:")
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=20, show_progress_bar=True)

# best_trial = study.best_trial

# print(f"\nbest trial number: {best_trial.number}")
# print(f"best trial value (NDCG@10): {best_trial.value:.4f}")
# print("best trial params:")
# for key, value in best_trial.params.items():
#     print(f"    {key}: {value}")

In [ ]:
optuna_params = {'factors': 32, 
                 'regularization': 0.019053374904390664, 
                 'iterations': 26,
                 'random_state': 42
                 }


In [ ]:
als_model = AlternatingLeastSquares(**optuna_params)
recsys_model = ImplicitModel(als_model)
recsys_model.fit(train_df)

In [ ]:
test_df_grouped['ials_preds_new'] = list(preds)

In [ ]:
preds = recsys_model.predict(test_df, top_k=top_recs)
ials_ndcg, ials_recall = evaluate_model(test_df_grouped, 'ials_preds_new', 'true_test_interactions', top_k=top_recs)

metrics_df = pd.DataFrame(
    data={
        'metrics': [ials_ndcg, ials_recall]
    },
    index=[f'NDCG@{top_recs}', f'Recall@{top_recs}']
)

metrics_df

## Get recommendations

In [ ]:
movies_data = os.path.join(movies_data_dir_path, 'movies.dat')

movies_cols_names = ['item_id', 'title', 'genres']
movies_df = pd.read_csv(movies_data, sep='::', engine='python', names=movies_cols_names, encoding='latin-1')
movies_df.head()

In [ ]:
def get_recommendations(users_ids, model, movies_df, top_k=top_recs):
    predict_df = pd.DataFrame({'user_id': users_ids})
    preds_ids = model.predict(predict_df, top_k=top_k)
    
    movie_map = movies_df.set_index('item_id')['title'].to_dict()
    
    result = []
    for user_id, user_preds in zip(users_ids, preds_ids):
        movie_titles = [movie_map.get(itm, f"unknown ID: {itm}") for itm in user_preds]
        
        result.append({
            'user_id': user_id,
            'recommendations': movie_titles
        })
        
    return pd.DataFrame(result)

In [ ]:
def print_recommendations(recs_df):
    print("Recommendations\n")
    
    for idx, row in recs_df.iterrows():
        user_id = row['user_id']
        movies = row['recommendations']
        
        print(f"User ID: {user_id}")
        print(f"Movies for you: {len(movies)}")
        
        for i, movie in enumerate(movies, start=1):
            print(f"  {i}. {movie}")
            
        print("-" * 50 + "\n")

In [ ]:
final_recs = get_recommendations([1, 10, 100], recsys_model, movies_df, top_k=5)
print_recommendations(final_recs)